# 01 — SQL avec `sqlite3`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer, interroger et modifier une base SQLite avec `sqlite3` stdlib
- maîtriser CREATE, SELECT, INSERT, UPDATE, DELETE
- écrire des JOIN et GROUP BY
- comprendre les transactions et l'utilisation de `with`
- **identifier et prévenir l'injection SQL** via requêtes paramétrées
- créer un index

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- classes, dataclasses, type hints
- context managers (`with`)
- fichiers, pathlib

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- SQLAlchemy (notebooks 02, 03)
- Alembic (notebook 04)

## Plan

1. Pourquoi SQL avant un ORM
2. Créer une base et une table
3. INSERT
4. SELECT, WHERE, ORDER BY
5. UPDATE et DELETE
6. JOIN
7. GROUP BY et agrégats
8. Transactions
9. **Injection SQL** et requêtes paramétrées
10. Index
11. Synthèse
12. Exercices

---

## 1. Pourquoi SQL avant un ORM

Un ORM génère du SQL. Pour diagnostiquer un bug de performance, un problème de migration ou un comportement inattendu, il faut **lire** le SQL qu'il produit. Maîtriser SQL directement est un **prérequis non négociable**.

---

## 2. Créer une base et une table

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')  # base en mémoire
conn.row_factory = sqlite3.Row       # résultats accessibles par nom

conn.execute('''
    CREATE TABLE salle (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nom TEXT NOT NULL UNIQUE,
        capacite INTEGER NOT NULL CHECK(capacite > 0)
    )
''')
conn.commit()


`':memory:'` crée une base temporaire. Pour persister, passer un chemin de fichier.

---

## 3. INSERT

In [ ]:
conn.execute('INSERT INTO salle (nom, capacite) VALUES (?, ?)', ('Mars', 12))
conn.execute('INSERT INTO salle (nom, capacite) VALUES (?, ?)', ('Venus', 6))
conn.execute('INSERT INTO salle (nom, capacite) VALUES (?, ?)', ('Io', 15))
conn.commit()


Le `?` est un **placeholder** : `sqlite3` le remplace par la valeur correspondante **de manière sécurisée**. On verra pourquoi en section 9 (injection SQL).

In [ ]:
# INSERT multiple avec executemany
salles = [('Europa', 10), ('Titan', 8), ('Ganymede', 20)]
conn.executemany('INSERT INTO salle (nom, capacite) VALUES (?, ?)', salles)
conn.commit()


---

## 4. SELECT, WHERE, ORDER BY

In [ ]:
rows = conn.execute('SELECT * FROM salle').fetchall()
for r in rows:
    print(r['id'], r['nom'], r['capacite'])


In [ ]:
conn.execute('SELECT * FROM salle WHERE capacite >= ?', (10,)).fetchall()


In [ ]:
conn.execute('SELECT * FROM salle ORDER BY capacite DESC').fetchall()


In [ ]:
conn.execute('SELECT * FROM salle ORDER BY nom LIMIT 3').fetchall()


---

## 5. UPDATE et DELETE

In [ ]:
conn.execute('UPDATE salle SET capacite = ? WHERE nom = ?', (14, 'Mars'))
conn.commit()
conn.execute('SELECT * FROM salle WHERE nom = ?', ('Mars',)).fetchone()['capacite']


In [ ]:
conn.execute('DELETE FROM salle WHERE nom = ?', ('Titan',))
conn.commit()
len(conn.execute('SELECT * FROM salle').fetchall())


---

## 6. JOIN

Créons une table de réservations et joignons-la aux salles.

In [ ]:
conn.execute('''
    CREATE TABLE reservation (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        salle_id INTEGER NOT NULL REFERENCES salle(id),
        creneau TEXT NOT NULL,
        organisateur TEXT NOT NULL
    )
''')
conn.executemany(
    'INSERT INTO reservation (salle_id, creneau, organisateur) VALUES (?, ?, ?)',
    [(1, 'lundi 9h', 'Alice'), (1, 'mardi 14h', 'Bob'), (2, 'lundi 9h', 'Charlie')],
)
conn.commit()


In [ ]:
rows = conn.execute('''
    SELECT s.nom, r.creneau, r.organisateur
    FROM reservation r
    JOIN salle s ON r.salle_id = s.id
    ORDER BY s.nom, r.creneau
''').fetchall()

for r in rows:
    print(dict(r))


---

## 7. GROUP BY et agrégats

In [ ]:
rows = conn.execute('''
    SELECT s.nom, COUNT(*) as nb_reservations
    FROM reservation r
    JOIN salle s ON r.salle_id = s.id
    GROUP BY s.nom
    ORDER BY nb_reservations DESC
''').fetchall()

for r in rows:
    print(dict(r))


---

## 8. Transactions

`sqlite3` ouvre une transaction implicite. `conn.commit()` la valide, `conn.rollback()` l'annule. Mieux : utiliser `with conn:` qui commit si tout va bien, rollback sinon.

In [ ]:
with conn:
    conn.execute('INSERT INTO salle (nom, capacite) VALUES (?, ?)', ('Callisto', 5))
    # si exception ici → rollback automatique

conn.execute('SELECT nom FROM salle WHERE nom = ?', ('Callisto',)).fetchone()['nom']


---

## 9. **Injection SQL** et requêtes paramétrées

C'est **le sujet le plus critique** de ce notebook. Une injection SQL permet à un attaquant de lire, modifier ou supprimer toute la base de données.

### ❌ Le code DANGEREUX

In [ ]:
# NE JAMAIS ÉCRIRE ÇA :
# nom_saisi = "'; DROP TABLE salle; --"
# conn.execute(f"SELECT * FROM salle WHERE nom = '{nom_saisi}'")
#
# → la requête devient :
# SELECT * FROM salle WHERE nom = ''; DROP TABLE salle; --'
# → la table est SUPPRIMÉE
print('⚠️  Ne jamais concaténer de valeur utilisateur dans du SQL !')


### ✅ Le code SÛR : requêtes paramétrées

In [ ]:
nom_saisi = "'; DROP TABLE salle; --"  # valeur malveillante
result = conn.execute('SELECT * FROM salle WHERE nom = ?', (nom_saisi,)).fetchone()
result  # None — pas de résultat, mais pas de destruction non plus


### Pourquoi ça fonctionne

Avec `?`, la valeur est transmise au moteur SQL comme une **donnée**, pas comme du **code**. Le moteur SQL ne l'interprète jamais comme des instructions SQL.

### Règle absolue

**JAMAIS de f-string, `.format()`, `%` ou `+` dans une requête SQL avec des valeurs utilisateur.** Toujours utiliser les **placeholders** (`?` pour sqlite3, `%s` pour psycopg2, `:param` pour SQLAlchemy).

---

## 10. Index

Un index accélère les recherches sur une colonne, au prix d'un stockage supplémentaire.

In [ ]:
conn.execute('CREATE INDEX idx_salle_nom ON salle (nom)')
conn.execute('CREATE INDEX idx_reservation_salle ON reservation (salle_id)')
conn.commit()


---

## Synthèse

| SQL | Rôle |
|---|---|
| `CREATE TABLE` | Définir une table |
| `INSERT INTO ... VALUES (?, ?)` | Ajouter une ligne |
| `SELECT ... WHERE ... ORDER BY` | Lire |
| `UPDATE ... SET ... WHERE` | Modifier |
| `DELETE FROM ... WHERE` | Supprimer |
| `JOIN ... ON` | Croiser deux tables |
| `GROUP BY ... HAVING` | Agrégation |
| `?` (placeholder) | **Sécurité anti-injection** |


### Règles à retenir

1. **Requêtes paramétrées TOUJOURS.** Pas de f-string dans du SQL. Jamais.
2. **`with conn:` pour les transactions** : commit auto, rollback auto sur exception.
3. **`row_factory = sqlite3.Row`** pour accéder par nom de colonne.
4. **SQL maîtrisé = ORM maîtrisé.** L'inverse n'est pas vrai.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — CRUD de base *(facile)*

Créer une base en mémoire avec une table `produit(id, nom, prix)`. Insérer 3 produits, sélectionner ceux avec prix > 10, mettre à jour un prix, supprimer un produit.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_SQL_avec_sqlite3", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import sqlite3

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.execute('CREATE TABLE produit (id INTEGER PRIMARY KEY, nom TEXT, prix REAL)')
conn.executemany('INSERT INTO produit VALUES (?, ?, ?)',
    [(1, 'pain', 1.2), (2, 'vin', 12.5), (3, 'fromage', 8.0)])
conn.commit()
for r in conn.execute('SELECT * FROM produit WHERE prix > ?', (10,)):
    print(dict(r))
conn.execute('UPDATE produit SET prix = ? WHERE nom = ?', (9.0, 'fromage'))
conn.execute('DELETE FROM produit WHERE nom = ?', ('pain',))
conn.commit()
for r in conn.execute('SELECT * FROM produit'):
    print(dict(r))
```

</details>

### Exercice 2 — JOIN et GROUP BY *(moyen)*

Créer `commande(id, produit_id, quantite)` liée à `produit`. Insérer des commandes. Écrire une requête qui renvoie le total de quantité par produit avec JOIN + GROUP BY.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_SQL_avec_sqlite3", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import sqlite3
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.execute('CREATE TABLE produit (id INTEGER PRIMARY KEY, nom TEXT, prix REAL)')
conn.execute('CREATE TABLE commande (id INTEGER PRIMARY KEY, produit_id INTEGER, quantite INTEGER)')
conn.executemany('INSERT INTO produit VALUES (?, ?, ?)', [(1, 'A', 10), (2, 'B', 20)])
conn.executemany('INSERT INTO commande VALUES (?, ?, ?)',
    [(1, 1, 5), (2, 1, 3), (3, 2, 7)])
conn.commit()
for r in conn.execute('''
    SELECT p.nom, SUM(c.quantite) as total
    FROM commande c JOIN produit p ON c.produit_id = p.id
    GROUP BY p.nom
'''):
    print(dict(r))
```

</details>

### Exercice 3 — Prouver l'injection SQL *(moyen)*

Écrire une fonction `rechercher_dangereuse(conn, nom)` qui utilise une f-string (volontairement mauvaise) et une `rechercher_safe(conn, nom)` qui utilise `?`. Prouver que la première est vulnérable à `"' OR '1'='1"`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_SQL_avec_sqlite3", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import sqlite3
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.execute('CREATE TABLE secret (id INTEGER PRIMARY KEY, code TEXT)')
conn.execute("INSERT INTO secret VALUES (1, 'TOP_SECRET')")
conn.commit()

def rechercher_dangereuse(conn: sqlite3.Connection, nom: str) -> list:
    # ❌ VOLONTAIREMENT MAUVAIS pour la démo
    return conn.execute(f"SELECT * FROM secret WHERE code = '{nom}'").fetchall()

def rechercher_safe(conn: sqlite3.Connection, nom: str) -> list:
    return conn.execute('SELECT * FROM secret WHERE code = ?', (nom,)).fetchall()

injection = "' OR '1'='1"
print('dangereux :', [dict(r) for r in rechercher_dangereuse(conn, injection)])
print('safe :', rechercher_safe(conn, injection))
```

</details>

### Exercice 4 — Réservation avec transaction (fil rouge) *(difficile)*

Écrire une fonction `reserver(conn, salle_nom: str, creneau: str, organisateur: str) -> int` qui :

1. vérifie que la salle existe (sinon `ValueError`) ;
2. vérifie que le créneau n'est pas pris (sinon `ValueError`) ;
3. insère la réservation **dans une transaction** ;
4. retourne l'id de la réservation.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_SQL_avec_sqlite3", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import sqlite3

def reserver(conn: sqlite3.Connection, salle_nom: str, creneau: str, org: str) -> int:
    row = conn.execute('SELECT id FROM salle WHERE nom = ?', (salle_nom,)).fetchone()
    if not row:
        raise ValueError(f'salle {salle_nom!r} inconnue')
    salle_id = row['id']
    existe = conn.execute(
        'SELECT 1 FROM reservation WHERE salle_id = ? AND creneau = ?',
        (salle_id, creneau)
    ).fetchone()
    if existe:
        raise ValueError(f'{salle_nom} déjà réservée à {creneau}')
    with conn:
        cur = conn.execute(
            'INSERT INTO reservation (salle_id, creneau, organisateur) VALUES (?, ?, ?)',
            (salle_id, creneau, org)
        )
        return cur.lastrowid  # type: ignore[return-value]

# (test avec la connexion du notebook)
print(reserver(conn, 'Mars', 'mercredi 10h', 'Diana'))
```

</details>

---

## Ressources externes

### Documentation officielle
- [`sqlite3` — stdlib](https://docs.python.org/3/library/sqlite3.html)
- [SQLite SQL syntax](https://www.sqlite.org/lang.html)

### Lectures complémentaires
- OWASP Top 10 — *SQL Injection* est la menace #3 toutes catégories confondues.